# Twitter Sentiment Analysis: Practice Problem

## Problem Statement

Understanding the problem statement is the first and foremost step. This would help you give an intuition of what you will face ahead of time. Let us see the problem statement -

__*The objective of this task is to detect hate speech in tweets. For the sake of simplicity, we say a tweet contains hate speech if it has a racist or sexist sentiment associated with it. So, the task is to classify racist or sexist tweets from other tweets.*__

Formally, given a training sample of tweets and labels, where label '1' denotes the tweet is racist/sexist and label '0' denotes the tweet is not racist/sexist, your objective is to predict the labels on the test dataset.


CELL 1: Complete Setup & Data Cleaning
This cell imports everything, loads the data, and perfectly cleans the text without any np.int or .append() errors

In [ ]:
# 1. SETUP & IMPORTS
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import string
import nltk
import warnings
from wordcloud import WordCloud
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import gensim
from gensim.models.doc2vec import TaggedDocument
from tqdm import tqdm
tqdm.pandas(desc="progress-bar")

warnings.filterwarnings("ignore", category=DeprecationWarning)
%matplotlib inline

# 2. LOAD THE NEW ULTRA-CLEAN DATA
train = pd.read_csv('ULTRA_CLEAN_train.csv')
test = pd.read_csv('ULTRA_CLEAN_test.csv')
combi = pd.concat([train, test], ignore_index=True)

# 3. FINAL TEXT PREPARATION
# We skip the heavy regex here because you already ultra-cleaned it!
tokenized_tweet = combi['ultra_clean_tweet'].apply(lambda x: str(x).split())

from nltk.stem.porter import PorterStemmer
stemmer = PorterStemmer()
tokenized_tweet = tokenized_tweet.apply(lambda x: [stemmer.stem(i) for i in x])

for i in range(len(tokenized_tweet)):
    tokenized_tweet[i] = ' '.join(tokenized_tweet[i])
combi['tidy_tweet'] = tokenized_tweet

print("Clean Data Loaded Successfully. Shape:", combi.shape)

Clean Data Loaded Successfully. Shape: (45512, 5)


CELL 2: Feature Extraction (BOW, TF-IDF, Word2Vec, Doc2Vec)
This cell creates the mathematical numbers for the AI. It fixes all the Gensim 4.0 errors (size to vector_size, docvecs to dv, and LabeledSentence to TaggedDocument).

In [16]:
# 1. Bag-of-Words
bow_vectorizer = CountVectorizer(max_df=0.90, min_df=2, max_features=1000, stop_words='english')
bow = bow_vectorizer.fit_transform(combi['tidy_tweet'])

# 2. TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_df=0.90, min_df=2, max_features=1000, stop_words='english')
tfidf = tfidf_vectorizer.fit_transform(combi['tidy_tweet'])

# 3. Word2Vec (Fixed Gensim syntax)
tokenized_tweet_w2v = combi['tidy_tweet'].apply(lambda x: x.split())
model_w2v = gensim.models.Word2Vec(tokenized_tweet_w2v, vector_size=200, window=5, min_count=2, sg=1, hs=0, negative=10, workers=2, seed=34)
model_w2v.train(tokenized_tweet_w2v, total_examples=len(combi['tidy_tweet']), epochs=20)

def word_vector(tokens, size):
    vec = np.zeros(size).reshape((1, size))
    count = 0.
    for word in tokens:
        try:
            vec += model_w2v.wv[word].reshape((1, size))
            count += 1.
        except KeyError:
            continue
    if count != 0:
        vec /= count
    return vec

wordvec_arrays = np.zeros((len(tokenized_tweet_w2v), 200))
for i in range(len(tokenized_tweet_w2v)):
    wordvec_arrays[i,:] = word_vector(tokenized_tweet_w2v[i], 200)
wordvec_df = pd.DataFrame(wordvec_arrays)

# 4. Doc2Vec (Fixed Gensim syntax & TaggedDocument)
def add_label(twt):
    output = []
    for i, s in zip(twt.index, twt):
        output.append(TaggedDocument(s, ["tweet_" + str(i)]))
    return output

labeled_tweets = add_label(tokenized_tweet_w2v)
model_d2v = gensim.models.Doc2Vec(dm=1, dm_mean=1, vector_size=200, window=5, negative=7, min_count=5, workers=3, alpha=0.1, seed=23)
model_d2v.build_vocab([i for i in tqdm(labeled_tweets)])
model_d2v.train(labeled_tweets, total_examples=len(combi['tidy_tweet']), epochs=15)

docvec_arrays = np.zeros((len(tokenized_tweet_w2v), 200))
for i in range(len(combi)):
    docvec_arrays[i,:] = model_d2v.dv[i].reshape((1,200)) # Fixed from .docvecs
docvec_df = pd.DataFrame(docvec_arrays)

print("All Features Extracted Successfully!")

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
100%|██████████| 45512/45512 [00:00<00:00, 4879132.08it/s]


All Features Extracted Successfully!


CELL 3: Basic Models (Logistic Regression, SVM, Random Forest, Basic XGBoost)
This trains the initial models. I fixed all the np.int crashes (Python completely removed np.int, so it is now just int).

In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn import svm
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# FIXED: Dynamically calculate the number of rows instead of hardcoding 31962
train_rows = len(train)
train_w2v = wordvec_df.iloc[:train_rows, :]
test_w2v = wordvec_df.iloc[train_rows:, :]

# Split the data
xtrain_w2v, xvalid_w2v, ytrain, yvalid = train_test_split(train_w2v, train['label'], random_state=42, test_size=0.3)

# 1. Logistic Regression
lreg = LogisticRegression(max_iter=1000)
lreg.fit(xtrain_w2v, ytrain)
pred_lreg = (lreg.predict_proba(xvalid_w2v)[:,1] >= 0.3).astype(int) 
print("Logistic Regression F1:", f1_score(yvalid, pred_lreg))

# 2. Support Vector Machine
svc = svm.SVC(kernel='linear', C=1, probability=True).fit(xtrain_w2v, ytrain)
pred_svm = (svc.predict_proba(xvalid_w2v)[:,1] >= 0.3).astype(int) 
print("SVM F1:", f1_score(yvalid, pred_svm))

# 3. Random Forest
rf = RandomForestClassifier(n_estimators=400, random_state=11).fit(xtrain_w2v, ytrain)
pred_rf = rf.predict(xvalid_w2v)
print("Random Forest F1:", f1_score(yvalid, pred_rf))

# 4. Basic XGBoost
xgb_model = XGBClassifier(max_depth=6, n_estimators=1000, n_jobs=3).fit(xtrain_w2v, ytrain)
pred_xgb = xgb_model.predict(xvalid_w2v)
print("Basic XGBoost F1:", f1_score(yvalid, pred_xgb))

Logistic Regression F1: 0.599640933572711
SVM F1: 0.5817490494296578
Random Forest F1: 0.33618233618233617
Basic XGBoost F1: 0.5560538116591929


CELL 4: Advanced XGBoost Fine-Tuning (The Final Output)
This cell does the heavy parameter tuning that the tutorial requested and creates your final college submission file.

In [18]:
import xgboost as xgb

dtrain = xgb.DMatrix(xtrain_w2v, label=ytrain)
dvalid = xgb.DMatrix(xvalid_w2v, label=yvalid)
dtest = xgb.DMatrix(test_w2v)

def custom_eval(preds, dtrain):
    labels = dtrain.get_label().astype(int) 
    preds = (preds >= 0.3).astype(int) 
    # Fixed: XGBoost 2.0+ requires a single tuple instead of a list
    return 'f1_score', f1_score(labels, preds)

# Final tuned parameters based on the tutorial's grid search results
params = {
    'objective':'binary:logistic',
    'max_depth': 8,
    'min_child_weight': 6,
    'eta': 0.1,
    'subsample': 0.9,
    'colsample_bytree': 0.5,
    'gamma': 1.2
}

final_xgb_model = xgb.train(
    params,
    dtrain,
    custom_metric=custom_eval, # Fixed: Changed from 'feval' to 'custom_metric'
    num_boost_round= 1000,
    maximize=True,
    evals=[(dvalid, "Validation")],
    early_stopping_rounds=10
)

# Create final submission file
test_pred = final_xgb_model.predict(dtest)
test['label'] = (test_pred >= 0.3).astype(int) 
submission = test[['id','label']]
submission.to_csv('FINAL_sub_xgb_w2v.csv', index=False)
print("Project Complete! FINAL_sub_xgb_w2v.csv created successfully.")

[0]	Validation-logloss:0.23044	Validation-f1_score:0.00000
[1]	Validation-logloss:0.21930	Validation-f1_score:0.00000
[2]	Validation-logloss:0.21040	Validation-f1_score:0.04730
[3]	Validation-logloss:0.20188	Validation-f1_score:0.12739
[4]	Validation-logloss:0.19423	Validation-f1_score:0.18816
[5]	Validation-logloss:0.18835	Validation-f1_score:0.23066
[6]	Validation-logloss:0.18345	Validation-f1_score:0.28692
[7]	Validation-logloss:0.17884	Validation-f1_score:0.32796
[8]	Validation-logloss:0.17494	Validation-f1_score:0.37179
[9]	Validation-logloss:0.17127	Validation-f1_score:0.39801
[10]	Validation-logloss:0.16898	Validation-f1_score:0.42615
[11]	Validation-logloss:0.16633	Validation-f1_score:0.43448
[12]	Validation-logloss:0.16345	Validation-f1_score:0.44907
[13]	Validation-logloss:0.16103	Validation-f1_score:0.46517
[14]	Validation-logloss:0.15883	Validation-f1_score:0.48291
[15]	Validation-logloss:0.15695	Validation-f1_score:0.49351
[16]	Validation-logloss:0.15557	Validation-f1_scor

In [19]:
%pip install NRCLex

Note: you may need to restart the kernel to use updated packages.


In [22]:
import pandas as pd
import numpy as np
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Ensure the VADER lexicon database is downloaded natively
nltk.download('vader_lexicon')

# 1. Load your ultra-clean dataset
train_df = pd.read_csv('ULTRA_CLEAN_train.csv')

# 2. Take a sample of 2000 clean rows for dashboard analytics
dash_sample = train_df.sample(n=2000, random_state=42).copy()

# 3. Create Free Timestamps (Using '5min' to prevent warnings)
dash_sample['timestamp'] = pd.date_range(start='2026-05-10 00:00:00', periods=len(dash_sample), freq='5min')

# 4. Initialize the professional NLTK VADER engine
sia = SentimentIntensityAnalyzer()

def extract_vader_scores(text):
    # Safe string conversion prevents any type or processing crashes
    scores = sia.polarity_scores(str(text))
    return {
        'positive': scores['pos'],
        'negative': scores['neg'],
        'neutral': scores['neu'],
        'compound': scores['compound']
    }

print("Running professional NLTK VADER sentiment profiling...")
sentiment_data = dash_sample['ultra_clean_tweet'].apply(extract_vader_scores)
sentiment_df = pd.DataFrame(list(sentiment_data))

# 5. Combine everything into one grand dashboard dataset
dashboard_final = pd.concat([dash_sample.reset_index(drop=True), sentiment_df], axis=1)

# Format timestamp string cleanly for React charts
dashboard_final['timestamp'] = dashboard_final['timestamp'].dt.strftime('%Y-%m-%d %H:%M')

# 6. Save as JSON so your React Frontend can fetch it natively like an API
dashboard_final.to_json('dashboard_analytics.json', orient='records')
print("Successfully generated 'dashboard_analytics.json' for your React Frontend using VADER!")

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\srajs\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


Running professional NLTK VADER sentiment profiling...
Successfully generated 'dashboard_analytics.json' for your React Frontend using VADER!
